# Probing Inside-Outside Structure in a Transformer Trained on the RHM

For a grammar of depth $L$ and a $2L$-layer transformer, the hypothesis is:
- **Layers 1…L (inside pass):** build level-$\ell$ representations bottom-up; representations remain ambiguous about the root.
- **Layers L+1…2L (outside pass):** propagate global context downward, eliminating spurious latents.

We test this by training linear probes at each layer to predict the ground-truth latents at every grammar level ($\ell = 1, 2, 3 = L$) from the residual stream.

**Probe dataset:** fully-ternary sequences generated on-the-fly from the grammar, so every token position has a well-defined parse-tree ancestor at every level.

**Expected signature of inside-outside:**
| Layer range | Root probe ($\ell=L$) | Level-1 probe ($\ell=1$) |
|---|---|---|
| 1…L | rising | high entropy / flat |
| L+1…2L | flat / saturated | **sharp rise** (outside resolves spurious latents) |


## 1. Setup & Imports

In [ ]:
import sys, os
import re
import math
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ── workspace root ───────────────────────────────────────────────────────────
WORKSPACE = Path("/Users/parley/Documents/Last_token")
sys.path.insert(0, str(WORKSPACE))

import models   # models/transformer.py → models.CLM

torch.manual_seed(0)
np.random.seed(0)
DEVICE = "mps"   # probing is lightweight; use "cuda" if available
print("PyTorch", torch.__version__, "| device:", DEVICE)


In [ ]:
# ── Grammar / dataset parameters ─────────────────────────────────────────────
# Adjust to match the training run you are probing.
V          = 8        # terminal vocabulary size
L          = 3         # grammar depth
BIN        = 2         # binary-rule multiplicity  (m_2 = BIN,  m_3 = BIN * V)
SEED_RULES = 12345678  # must match the training run

# ── Model hyper-parameters ────────────────────────────────────────────────────
# These cannot be inferred from the state-dict; set them to match training.
NUM_HEADS  = 16
# (embedding_dim, ffwd_size, depth are inferred automatically below)

# ── Probe dataset ─────────────────────────────────────────────────────────────
N_PROBE      = 32768   # number of fully-ternary sequences
PROBE_SEED   = 42

# ── Checkpoint ────────────────────────────────────────────────────────────────
SINGLE_CHECKPOINT = WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t512.pt"

print("Grammar  : v=%d, L=%d, bin=%d, seed_rules=%d" % (V, L, BIN, SEED_RULES))
print("Probe set: N=%d, seed=%d" % (N_PROBE, PROBE_SEED))
print("Checkpoint exists:", SINGLE_CHECKPOINT.exists())


## 2. Grammar Reconstruction & Probe Dataset

We reconstruct the grammar from the known seed, then generate **fully-ternary** sequences and track the ground-truth latent at every grammar level.

For a depth-$L$ all-ternary tree with $v^L = 27$ leaves:
- Token at input position $i$ has level-1 parent $\lfloor i/3 \rfloor$, level-2 parent $\lfloor i/9 \rfloor$, and root parent 0.


In [ ]:
_rhm_dir = str(WORKSPACE / "datasets" / "rhm")
if _rhm_dir not in sys.path:
    sys.path.insert(0, _rhm_dir)

from mixed_random_hierarchy_model import (
    sample_padded_rules,
    create_probabilities,
    sample_data_from_labels_varying_tree_tensorized,
)

m_2 = BIN          # binary-rule multiplicity
m_3 = BIN * V      # ternary-rule multiplicity
S_2, S_3 = 2, 3
D_MAX    = S_3 ** L        # max sequence length (27 for L=3, ternary)
T        = D_MAX - 1       # transformer block_size (26)
N_CLASSES = V              # num_classes == num_features

rules = sample_padded_rules(V, N_CLASSES, m_2, m_3, S_2, S_3, L, SEED_RULES)

# Ternary-only probabilities (put all weight on ternary rules)
ternary_probs = {}
for l in range(L):
    ternary_probs[l] = torch.cat([
        torch.zeros(m_2),
        torch.full((m_3,), 1.0 / m_3)
    ])

print(f"Grammar: v={V}, L={L}, m_2={m_2}, m_3={m_3}, d_max={D_MAX}, T={T}")
print(f"n_classes (root vocab) = {N_CLASSES}")


In [ ]:
def generate_mixed_with_last_branch_latents(
    N, rules, V, L, m_2, m_3, probs, d_max, seed=42
):
    """
    Generate N mixed (binary+ternary) sequences and record the ground-truth
    latent on the path to the last (rightmost) real leaf at every grammar level.

    The 'last branch' is the path from the root to the rightmost leaf in each
    parse tree. At every internal level, the latent recorded is the node that
    is the direct ancestor of that rightmost leaf at that level.

    Parameters
    ----------
    probs : dict {l: Tensor(m_2+m_3)}
        Rule probability distribution.
    d_max : int
        Length of the padded output sequences (= 3^L).

    Returns
    -------
    sequences : LongTensor (N, d_max)
        Terminal sequences with real tokens right-aligned; unused positions
        on the left are filled with the fake symbol V.
    last_latents : list of L LongTensors, each of shape (N,)
        last_latents[0]   : root label (level L)
        last_latents[1]   : level L-1 latent on the last branch
        ...
        last_latents[L-1] : level-1 latent on the last branch
    """
    torch.manual_seed(seed)

    root_labels = torch.randint(0, V, (N,))
    features = root_labels.clone()
    last_latents = [root_labels.clone()]

    for l in range(L - 1):
        flat = features.reshape(-1)

        chosen = torch.multinomial(
            probs[l].unsqueeze(0).expand(flat.numel(), -1),
            num_samples=1,
            replacement=True,
        ).squeeze(-1).reshape(features.shape)

        features = rules[l][features, chosen].flatten(start_dim=1)

        # Find the last real node in each row.
        is_real = features != V
        last_idx = (
            (features.shape[1] - 1)
            - is_real.flip(dims=[1]).long().argmax(dim=1)
        )

        last_latents.append(
            features[torch.arange(N, device=features.device), last_idx].clone()
        )

    # Final expansion: level-1 nodes -> terminal tokens
    flat = features.reshape(-1)

    chosen = torch.multinomial(
        probs[L - 1].unsqueeze(0).expand(flat.numel(), -1),
        num_samples=1,
        replacement=True,
    ).squeeze(-1).reshape(features.shape)

    terminals = rules[L - 1][features, chosen].flatten(start_dim=1)

    # Pack real tokens to the right, leaving V padding on the left.
    # Stable sorting preserves the original order of the real tokens.
    is_real = terminals != V
    sort_idx = is_real.long().argsort(
        dim=1,
        descending=False,
        stable=True,
    )
    sequences = terminals.gather(1, sort_idx)

    return sequences, last_latents


# Mixed rule probabilities: equal probability mass on binary and ternary rules
#mixed_probs = create_probabilities(m_2, m_3, L)

sequences_probe, last_latents = generate_mixed_with_last_branch_latents(
    N_PROBE,
    rules,
    V,
    L,
    m_2,
    m_3,
    ternary_probs,
    D_MAX,
    seed=PROBE_SEED,
)

seq_lengths = (sequences_probe != V).sum(dim=1)

print(f"sequences_probe shape : {sequences_probe.shape}")
print(
    f"sequence length : min={seq_lengths.min().item()}, "
    f"max={seq_lengths.max().item()}, "
    f"mean={seq_lengths.float().mean().item():.1f}"
)

for i, lat in enumerate(last_latents):
    print(f"last_latents[{i}] (level {L - i}) shape: {lat.shape}")

In [ ]:
sequences_probe

In [ ]:
# probe_inputs: first T tokens of each padded sequence, fed to the transformer.
# Block-size T = D_MAX - 1 matches training; sequences are already D_MAX long.
probe_inputs = sequences_probe[:, :T].long()   # (N, T)

# Because sequences are right-aligned and column D_MAX-1 is held out, column T-1
# is the final observed real token for every topology. Its residual stream is the
# causal representation used to predict the held-out token.
context_pos = T - 1

# ── Ground-truth last-branch latents (one label per sequence) ─────────────────
# last_latents[0] : root       (level L)
# last_latents[1] : level L-1  ancestor of the rightmost leaf
# last_latents[L-1]: level 1   direct parent of the rightmost leaf
y_root   = last_latents[0].numpy()          # (N,)
y_level2 = last_latents[1].numpy()          # (N,)
y_level1 = last_latents[L - 1].numpy()      # (N,)

print(f"probe_inputs      : {probe_inputs.shape}")
print(f"causal context position: {context_pos}")
print(f"y_root   range: [{y_root.min()}, {y_root.max()}]  "
      f"({len(np.unique(y_root))} unique)")
print(f"y_level2 range: [{y_level2.min()}, {y_level2.max()}]")
print(f"y_level1 range: [{y_level1.min()}, {y_level1.max()}]")


## 3. Model Loading from Checkpoint

We infer most architecture parameters from the state-dict shapes. Only `num_heads` must be set manually (it does not appear in any tensor shape).


In [ ]:
def infer_arch(state_dict, num_heads):
    """Infer CLM constructor kwargs from a state_dict.  num_heads must be supplied."""
    vocab_size    = state_dict["token_embedding_table.weight"].shape[0]
    embedding_dim = state_dict["token_embedding_table.weight"].shape[1]
    block_size    = state_dict["blocks.0.attn.tril"].shape[0]
    num_layers    = max(
        int(k.split(".")[1]) for k in state_dict if k.startswith("blocks.")
    ) + 1
    ffwd_size     = (state_dict["blocks.0.ffwd.net.0.weight"].shape[0] // embedding_dim)
    rope          = "position_embedding_table.weight" not in state_dict
    share_emb     = (
        state_dict["token_embedding_table.weight"].data_ptr()
        == state_dict.get("lm_head.weight", torch.empty(0)).data_ptr()
    )
    return dict(
        vocab_size    = vocab_size,
        block_size    = block_size,
        embedding_dim = embedding_dim,
        num_heads     = num_heads,
        ffwd_size     = ffwd_size,
        num_layers    = num_layers,
        dropout       = 0.0,
        rope          = rope,
        share_emb     = False,
    )


def load_model_from_checkpoint(ckpt_path, num_heads, device="cpu"):
    """Load a CLM from a checkpoint file produced by main.py."""
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt["model"]
    arch = infer_arch(state_dict, num_heads)
    model = models.CLM(**arch)
    model.load_state_dict(state_dict)
    model.to(device).eval()
    step  = ckpt.get("step", None)
    state = ckpt.get("state", {})
    return model, arch, step, state


# ── Use SINGLE_CHECKPOINT ─────────────────────────────────────────────────────
all_ckpts = [SINGLE_CHECKPOINT] if SINGLE_CHECKPOINT.exists() else []
print(f"Found {len(all_ckpts)} checkpoint files")
if all_ckpts:
    print("Checkpoint:", all_ckpts[0].name)


## 4. Activation Extraction via Forward Hooks

We register a hook on each `TransformerBlock` to capture the residual stream **after** that block (post-attention + post-FFN + skip connection). We also capture the embedding output (layer 0).

The result is a tensor of shape `(num_layers+1, N, T, embedding_dim)`.


In [ ]:
@torch.no_grad()
def extract_activations(model, inputs, device="cpu", batch_size=256):
    """
    Run `inputs` through `model` and return the residual stream at every layer.

    Parameters
    ----------
    model   : CLM (eval mode, frozen)
    inputs  : LongTensor (N, T)
    device  : str
    batch_size : int  — process in mini-batches to avoid OOM

    Returns
    -------
    activations : ndarray (num_probe_points, N, T, D)
        Index 0   = after token+position embedding (before any block)
        Index k>0 = after TransformerBlock k-1  (residual stream)
    """
    model.eval()
    num_layers = model.num_layers
    D          = model.embedding_dim
    N, Tseq    = inputs.shape
    num_points = num_layers + 1        # embedding + L blocks

    all_acts = [[] for _ in range(num_points)]   # list-of-lists

    for start in range(0, N, batch_size):
        batch = inputs[start:start + batch_size].to(device)
        B     = batch.shape[0]

        captured = {}   # hook_idx -> (B, T, D) tensor

        hooks = []

        # Hook to grab the embedding output (before any block)
        def _emb_hook(module, inp, out):
            captured[0] = out.detach().cpu()

        # We patch the forward pass manually to avoid modifying the model.
        # Strategy: temporarily replace model.blocks with a hooked Sequential.
        block_outputs = {}

        def make_block_hook(idx):
            def _hook(module, inp, out):
                block_outputs[idx] = out.detach().cpu()
            return _hook

        block_hooks = [
            block.register_forward_hook(make_block_hook(i))
            for i, block in enumerate(model.blocks)
        ]

        # Run forward pass; capture embedding separately via a hook on blocks[0]'s input
        # We intercept the embedding by hooking the ln_f or the first block input.
        # Simplest: hook the FIRST block's __call__ *input*.
        def _first_block_input_hook(module, inp, out):
            # inp[0] is the input to the first block = embedding output
            captured[0] = inp[0].detach().cpu()

        first_block_hook = model.blocks[0].register_forward_pre_hook(
            lambda m, inp: captured.update({0: inp[0].detach().cpu()})
        )

        _ = model(batch)   # forward pass triggers all hooks

        first_block_hook.remove()
        for h in block_hooks:
            h.remove()

        # Store results
        all_acts[0].append(captured[0])          # embedding
        for i in range(num_layers):
            all_acts[i + 1].append(block_outputs[i])  # after block i

    # Concatenate batches
    result = np.stack(
        [np.concatenate([a.numpy() for a in layer_acts], axis=0) for layer_acts in all_acts],
        axis=0,
    )  # (num_points, N, T, D)
    return result


# ── Sanity check on one checkpoint ───────────────────────────────────────────
if all_ckpts:
    _model, _arch, _step, _state = load_model_from_checkpoint(
        all_ckpts[-1], NUM_HEADS, device=DEVICE
    )
    print("Architecture:", {k: v for k, v in _arch.items() if k != "dropout"})
    print("Checkpoint step:", _step, " | val_loss:", _state.get("val_loss", "?"))

    _acts = extract_activations(_model, probe_inputs, device=DEVICE, batch_size=256)
    print("Activations shape (n_probe_points, N, T, D):", _acts.shape)
    del _model


## 5. Linear Probes

For each probe point (layer index) we train a `LogisticRegression` on the flattened `(N*T, D)` activations to predict each of the three latent levels.

We use a `StandardScaler → LogisticRegression(C=10, max_iter=1000)` pipeline, which is standard in probing literature. No regularisation tuning needed for this clean synthetic task.


In [ ]:
def train_probes(activations, y_root, y_level2, y_level1, test_frac=0.3, random_state=0):
    """
    Train a linear probe at each probe point (layer).

    Parameters
    ----------
    activations : ndarray (n_points, N, T, D)
    y_*         : ndarray (N*T,)  — ground-truth labels (flattened over sequences × positions)

    Returns
    -------
    results : dict with keys "root", "level2", "level1"
              each value is a list of length n_points containing probe accuracy (float)
    """
    n_points, N, Tseq, D = activations.shape
    NT = N * Tseq

    # Train / test split (positional — avoids data leakage between sequences)
    n_test  = int(NT * test_frac)
    n_train = NT - n_test
    idx     = np.random.RandomState(random_state).permutation(NT)
    train_idx, test_idx = idx[:n_train], idx[n_train:]

    results = {"root": [], "level2": [], "level1": []}
    label_map = {"root": y_root, "level2": y_level2, "level1": y_level1}

    for k in range(n_points):
        X = activations[k].reshape(NT, D)   # (NT, D)
        train_accs = {}
        for name, y in label_map.items():
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf",    LogisticRegression(C=10, max_iter=5000,
                                              solver="lbfgs", random_state=0,
                                              n_jobs=-1)),
            ])
            pipe.fit(X[train_idx], y[train_idx])
            train_accs[name] = pipe.score(X[train_idx], y[train_idx])
            acc = pipe.score(X[test_idx], y[test_idx])
            results[name].append(acc)

        print(f"  layer {k:2d} | "
              f"root={results['root'][-1]:.3f}(tr={train_accs['root']:.3f})  "
              f"lv2={results['level2'][-1]:.3f}(tr={train_accs['level2']:.3f})  "
              f"lv1={results['level1'][-1]:.3f}(tr={train_accs['level1']:.3f})")

    return results


# Chance-level baselines
chance_root   = 1.0 / N_CLASSES
chance_level2 = 1.0 / N_CLASSES   # level-2 latents are also drawn from V classes
chance_level1 = 1.0 / N_CLASSES
print(f"Chance level (all): {chance_root:.3f}  (1/{N_CLASSES})")


## 6. End-of-Training Probe: Single Checkpoint

Run the probing experiment on the **final** checkpoint to establish the *spatial* inside-outside structure.


In [ ]:
assert all_ckpts, "No checkpoint files found — check CHECKPOINT_DIR and CHECKPOINT_GLOB."

final_ckpt = all_ckpts[-1]
print(f"Probing checkpoint: {final_ckpt.name}")

model_final, arch_final, step_final, state_final = load_model_from_checkpoint(
    final_ckpt, NUM_HEADS, device=DEVICE
)
num_layers = arch_final["num_layers"]
print(f"num_layers={num_layers}, step={step_final}, val_loss={state_final.get('val_loss','?'):.4f}")

acts_final = extract_activations(model_final, probe_inputs, device=DEVICE, batch_size=256)
del model_final   # free memory

#print(f"\nTraining probes on activations {acts_final.shape} ...")
#probe_results_final = train_probes(acts_final, y_root, y_level2, y_level1)


## 8. Token-level probe: predict the held-out final token

Each mixed-topology sentence is right-aligned to `D_MAX`. The model receives `sequence[:, :-1]`, while `sequence[:, -1]` is never included in the input and is used as the token label. Thus the residual stream at input position `T-1` is the causal representation immediately before the masked/held-out last token.

The split is made over sentences (not token positions), and the same split is reused at every layer. This measures how linearly decodable the next-token identity is throughout the network without leaking examples between train and test sets.

In [ ]:
# The target is genuinely held out: probe_inputs contains columns [0, D_MAX-2].
y_token = sequences_probe[:, -1].cpu().numpy()
assert probe_inputs.shape[1] == sequences_probe.shape[1] - 1
assert np.all(y_token != V), "The held-out column must contain a real token"

TOKEN_PROBE_C = 0.1


def train_token_probe(
    activations, y_token, context_position=-1, test_frac=0.3,
    random_state=0, C=TOKEN_PROBE_C,
):
    """Train one multinomial linear probe per residual-stream depth."""
    n_points, N, _, _ = activations.shape
    if len(y_token) != N:
        raise ValueError(f"Expected {N} token labels, got {len(y_token)}")

    rng = np.random.RandomState(random_state)
    order = rng.permutation(N)
    n_test = max(1, int(N * test_frac))
    test_idx, train_idx = order[:n_test], order[n_test:]

    results = {"train_accuracy": [], "test_accuracy": []}
    for k in range(n_points):
        X = activations[k, :, context_position, :]
        probe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=C, max_iter=5000, solver="lbfgs", random_state=random_state
            )),
        ])
        probe.fit(X[train_idx], y_token[train_idx])
        train_acc = probe.score(X[train_idx], y_token[train_idx])
        test_acc = probe.score(X[test_idx], y_token[test_idx])
        results["train_accuracy"].append(train_acc)
        results["test_accuracy"].append(test_acc)
        print(f"  layer {k:2d} | token={test_acc:.3f} (train={train_acc:.3f})")

    results["chance_uniform"] = 1.0 / V
    counts = np.bincount(y_token[train_idx], minlength=V)
    results["chance_majority"] = counts.max() / counts.sum()
    results["train_indices"] = train_idx
    results["test_indices"] = test_idx
    return results


print("Held-out token counts:", np.bincount(y_token, minlength=V))
probe_results_token = train_token_probe(acts_final, y_token)

In [ ]:
import pickle

output_path = WORKSPACE / f"probe_results_ternary_step_{step_final}_{N_PROBE}_C_01_token_level.pkl"
with open(output_path, "wb") as f:
    pickle.dump({"token_probe": probe_results_token}, f)

print("Saved token-level probe results to", output_path)

In [ ]:
layer_labels = ["emb"] + [f"L{k + 1}" for k in range(num_layers)]
x = np.arange(len(layer_labels))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(x, probe_results_token["test_accuracy"], "o-", lw=2, label="test")
ax.plot(x, probe_results_token["train_accuracy"], "o--", alpha=0.55, label="train")
ax.axhline(probe_results_token["chance_uniform"], color="grey", ls=":", label=f"uniform chance (1/{V})")
ax.axhline(probe_results_token["chance_majority"], color="tab:red", ls=":", label="train-majority baseline")
ax.set_xticks(x, layer_labels)
ax.set_ylim(0, 1.02)
ax.set_xlabel("Residual-stream depth")
ax.set_ylabel("Held-out final-token accuracy")
ax.set_title(f"Token-level linear probe — checkpoint step {step_final}")
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
step_final


## 9. Compare token-probe training accuracy across checkpoints

Load the saved token-level probe results for steps 5,792, 262,144, and 1,048,576 and overlay their training accuracies against residual-stream depth. Depth 0 is the embedding representation; depth $k$ is the output of transformer block $k$.

In [ ]:
STEPS_TO_COMPARE = [5792, 262144, 1482912]
comparison_results = {}
result_types = {
    "VT training": "",
    "FT training": "_ternary_checkpoints",
}
import pickle

for step in STEPS_TO_COMPARE:
    for result_type, suffix in result_types.items():
        result_path = WORKSPACE / (
            f"probe_results_ternary_step_{step}_{N_PROBE}_C_01_token_level{suffix}.pkl"
        )
        if not result_path.exists():
            raise FileNotFoundError(f"Missing token-probe results: {result_path}")
        with open(result_path, "rb") as f:
            saved = pickle.load(f)

        # Accept both wrapped and directly saved token-probe dictionaries.
        token_result = saved["token_probe"] if "token_probe" in saved else saved
        comparison_results[(step, result_type)] = np.asarray(
            token_result["test_accuracy"], dtype=float
        )

n_depths = {len(values) for values in comparison_results.values()}
if len(n_depths) != 1:
    raise ValueError(f"Checkpoints have inconsistent numbers of probe depths: {n_depths}")

depth = np.arange(n_depths.pop())
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7.5, 4.8))
colors = dict(zip(STEPS_TO_COMPARE, plt.rcParams["axes.prop_cycle"].by_key()["color"]))
step_labels = {
    5792: r"$\ell=1$ ($t=5{,}792$)",
    262144: r"$\ell=2$ ($t=262{,}144$)",
    1048576: r"$\ell=3$ ($t=1{,}048{,}576$)",
}
styles = {
    # Wide, translucent underlay: remains visible through gaps in the dashed curve.
    "VT training": {
        "linestyle": "-", "marker": "o", "linewidth": 3.2,
        "markersize": 7, "alpha": 0.55, "zorder": 2,
    },
    # Narrow, hollow, dashed overlay: distinguishable even at exact overlap.
    "FT training": {
        "linestyle": (0, (4, 2)), "marker": "s", "linewidth": 1.8,
        "markersize": 6, "markerfacecolor": "white",
        "markeredgewidth": 1.5, "alpha": 1.0, "zorder": 3,
    },
}
for (step, result_type), test_accuracy in comparison_results.items():
    ax.plot(
        depth, test_accuracy, color=colors[step],
        **styles[result_type],
    )

ax.axhline(1.0 / V, color="grey", linestyle=":", linewidth=1.5, zorder=1)
ax.set_xticks(depth, ["emb"] + [f"L{k}" for k in depth[1:]])
ax.set_ylim(0, 1.02)
ax.set_xlabel("Transformer depth")
ax.set_ylabel("Token-probe training accuracy")
ax.set_title("Held-out final-token linear probe across training checkpoints")
ax.grid(alpha=0.2)
step_handles = [
    Line2D([0], [0], color=colors[step], linewidth=3, label=step_labels[step])
    for step in STEPS_TO_COMPARE
]
type_handles = [
    Line2D([0], [0], color="black", **styles[result_type], label=result_type)
    for result_type in result_types
]
type_handles.append(
    Line2D([0], [0], color="grey", linestyle=":", linewidth=1.5,
           label=f"uniform chance (1/{V})")
)
step_legend = ax.legend(handles=step_handles, title="Level / training step", loc="lower right")
ax.add_artist(step_legend)
ax.legend(handles=type_handles, title="Curve type", loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7.5, 4.8))

colors = dict(
    zip(
        STEPS_TO_COMPARE,
        plt.rcParams["axes.prop_cycle"].by_key()["color"]
    )
)

step_labels = {
    5792: r"$\ell=1$ ($t=5{,}792$)",
    262144: r"$\ell=2$ ($t=262{,}144$)",
    1048576: r"$\ell=3$ ($t=1{,}048{,}576$)",
}

styles = {
    "VT training": {
        "linestyle": "-",
        "marker": "o",
        "linewidth": 3.2,
        "markersize": 7,
        "alpha": 0.55,
        "zorder": 2,
    },
    "FT training": {
        "linestyle": (0, (4, 2)),
        "marker": "s",
        "linewidth": 1.8,
        "markersize": 6,
        "markerfacecolor": "white",
        "markeredgewidth": 1.5,
        "alpha": 1.0,
        "zorder": 3,
    },
}

# ---------------------------------------------------------------------
# Empirical probe curves
# ---------------------------------------------------------------------

for (step, result_type), test_accuracy in comparison_results.items():
    ax.plot(
        depth,
        test_accuracy,
        color=colors[step],
        **styles[result_type],
    )

# ---------------------------------------------------------------------
# Theoretical posterior plateaus
# ---------------------------------------------------------------------

theoretical_accuracy = {
    "bottom": 0.442,
    "bottom-intermediate": 0.737,
    "all": 0.851,
}

theory_colors = {
    "bottom": colors[5792],
    "bottom-intermediate": colors[262144],
    "all": colors[1048576],
}

# Short horizontal segments just to the right of the final transformer layer.
x_right = depth[-1]
x_start = x_right + 0.1
x_end = x_right + 0.4

for known_layers, accuracy in theoretical_accuracy.items():
    ax.hlines(
        y=accuracy,
        xmin=x_start,
        xmax=x_end,
        color=theory_colors[known_layers],
        linestyle="-",
        linewidth=2.6,
        zorder=4,
    )

# ---------------------------------------------------------------------
# Chance level and axes
# ---------------------------------------------------------------------

ax.axhline(
    1.0 / V,
    color="grey",
    linestyle=":",
    linewidth=1.5,
    zorder=1,
)

ax.set_xticks(
    depth,
    ["emb"] + [f"L{k}" for k in depth[1:]],
)

# Leave room on the right for the theoretical plateau segments.
ax.set_xlim(depth[0] - 0.2, x_end + 0.1)
ax.set_ylim(0, 1.02)

ax.set_xlabel("Transformer depth")
ax.set_ylabel("Token-probe training accuracy")
ax.set_title("Held-out final-token linear probe across training checkpoints")
ax.grid(alpha=0.2)

# ---------------------------------------------------------------------
# Legends
# ---------------------------------------------------------------------

step_handles = [
    Line2D(
        [0],
        [0],
        color=colors[step],
        linewidth=3,
        label=step_labels[step],
    )
    for step in STEPS_TO_COMPARE
]

type_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        **styles[result_type],
        label=result_type,
    )
    for result_type in result_types
]

type_handles += [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        linewidth=2.6,
        label="theory: posterior",
    )
    # Line2D(
    #     [0],
    #     [0],
    #     color="grey",
    #     linestyle=":",
    #     linewidth=1.5,
    #     label=f"uniform chance (1/{V})",
    # ),
]

step_legend = ax.legend(
    handles=step_handles,
    title="Level / training step",
    loc="lower right",
)
ax.add_artist(step_legend)

ax.legend(
    handles=type_handles,
    title="Curve type",
    loc="lower left",
)

plt.tight_layout()
plt.show()

In [ ]:
STEPS_TO_COMPARE = [5792, 262144, 1048576]
comparison_results = {}
import pickle

for step in STEPS_TO_COMPARE:
    result_path = WORKSPACE / (
        f"probe_results_step_{step}_{N_PROBE}_C_01_token_level.pkl"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            f"Missing token-probe results: {result_path}"
        )

    with open(result_path, "rb") as f:
        saved = pickle.load(f)

    # New files contain both latent and token probes.
    # This fallback also accepts a token-probe dictionary saved directly.
    token_result = saved["token_probe"] if "token_probe" in saved else saved

    comparison_results[step] = np.asarray(
        token_result["test_accuracy"],
        dtype=float,
    )


# ---------------------------------------------------------------------
# Check number of transformer depths
# ---------------------------------------------------------------------
n_depths = {
    len(values)
    for values in comparison_results.values()
}

if len(n_depths) != 1:
    raise ValueError(
        f"Checkpoints have inconsistent numbers of probe depths: {n_depths}"
    )

depth = np.arange(n_depths.pop())


# ---------------------------------------------------------------------
# Theoretical plateaus
# ---------------------------------------------------------------------
theoretical_plateaus = {
    "bottom": 0.45026666666666665,
    "bottom-intermediate": 0.7340166666666667,
    "all": 0.8502833333333333,
}


# ---------------------------------------------------------------------
# Plot
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 4.8))

for step, test_accuracy in comparison_results.items():
    ax.plot(
        depth,
        test_accuracy,
        marker="o",
        linewidth=2,
        label=f"step {step:,}",
    )


# ---------------------------------------------------------------------
# Theoretical plateau lines
# ---------------------------------------------------------------------
ax.axhline(
    theoretical_plateaus["bottom"],
    linestyle="--",
    linewidth=1.8,
    label=rf"theory: bottom ({theoretical_plateaus['bottom']:.3f})",
)

ax.axhline(
    theoretical_plateaus["bottom-intermediate"],
    linestyle="--",
    linewidth=1.8,
    label=(
        rf"theory: bottom-intermediate "
        rf"({theoretical_plateaus['bottom-intermediate']:.3f})"
    ),
)

ax.axhline(
    theoretical_plateaus["all"],
    linestyle="--",
    linewidth=1.8,
    label=rf"theory: all ({theoretical_plateaus['all']:.3f})",
)


# Uniform chance
ax.axhline(
    1.0 / V,
    color="grey",
    linestyle=":",
    linewidth=1.5,
    label=f"uniform chance (1/{V})",
)


# ---------------------------------------------------------------------
# Formatting
# ---------------------------------------------------------------------
ax.set_xticks(
    depth,
    ["emb"] + [f"L{k}" for k in depth[1:]],
)

ax.set_ylim(0, 1.02)

ax.set_xlabel("Transformer depth")
ax.set_ylabel("Token-probe test accuracy")

ax.set_title(
    "Held-out final-token linear probe across training checkpoints"
)

ax.grid(alpha=0.2)

ax.legend(
    title="Checkpoint / theory",
    fontsize=9,
)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.lines import Line2D

step_labels = {
    5792: r"$\ell=1$ ($t=5{,}792$)",
    262144: r"$\ell=2$ ($t=262{,}144$)",
    1482912: r"$\ell=3$ ($t=1{,}482{,}912$)",
}

# ---------------------------------------------------------------------
# Plot: two side-by-side panels
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 4.8),
    sharey=True,
)

panel_info = [
    ("VT training: ternary", "VT training: ternary topology"),
    ("VT training: all topologies", "VT training: all topologies"),
]

# Short theoretical plateau segments just to the right of final layer
x_right = depth[-1]
x_start = x_right + 0.1
x_end = x_right + 0.4

for ax, (result_type, panel_title) in zip(axes, panel_info):

    # -----------------------------------------------------------------
    # Empirical probe curves
    # -----------------------------------------------------------------
    for step in STEPS_TO_COMPARE:
        test_accuracy = comparison_results[(step, result_type)]

        ax.plot(
            depth,
            test_accuracy,
            color=colors[step],
            **styles[result_type],
        )

    # -----------------------------------------------------------------
    # Theoretical posterior plateaus
    # -----------------------------------------------------------------
    for known_layers, accuracy in theoretical_accuracy.items():
        ax.hlines(
            y=accuracy,
            xmin=x_start,
            xmax=x_end,
            color=theory_colors[known_layers],
            linestyle="-",
            linewidth=2.6,
            zorder=4,
        )

    # -----------------------------------------------------------------
    # Chance level
    # -----------------------------------------------------------------
    ax.axhline(
        1.0 / V,
        color="grey",
        linestyle=":",
        linewidth=1.5,
        zorder=1,
    )

    # -----------------------------------------------------------------
    # Axes formatting
    # -----------------------------------------------------------------
    ax.set_xticks(
        depth,
        ["emb"] + [f"L{k}" for k in depth[1:]],
    )

    ax.set_xlim(depth[0] - 0.2, x_end + 0.1)
    ax.set_ylim(0, 1.02)

    ax.set_xlabel("Transformer depth")
    ax.set_title(panel_title)

    ax.grid(alpha=0.2)

axes[0].set_ylabel("Token-probe test accuracy")


# ---------------------------------------------------------------------
# Shared legend
# ---------------------------------------------------------------------
step_handles = [
    Line2D(
        [0],
        [0],
        color=colors[step],
        linewidth=3,
        label=step_labels[step],
    )
    for step in STEPS_TO_COMPARE
]

step_handles += [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        linewidth=2.6,
        label="theory: posterior",
    ),
]

fig.legend(
    handles=step_handles,
    title="Level / training step",
    loc="center right",
    bbox_to_anchor=(1.01, 0.5),
)

fig.suptitle(
    "Held-out final-token linear probe across training checkpoints",
    y=1.02,
)

plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------------------------------------------------
# Checkpoints to compare
# ---------------------------------------------------------------------
STEPS_TO_COMPARE = [
    512,
    5792,
    262144,
    1482912,
]

step_labels = {
    512: r"$\ell=0$ ($t=512$)",
    5792: r"$\ell=1$ ($t=5{,}792$)",
    262144: r"$\ell=2$ ($t=262{,}144$)",
    1482912: r"$\ell=3$ ($t=1{,}482{,}912$)",
}

# ---------------------------------------------------------------------
# Assign one color to each checkpoint
# ---------------------------------------------------------------------
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

colors = {
    step: color_cycle[i % len(color_cycle)]
    for i, step in enumerate(STEPS_TO_COMPARE)
}

# ---------------------------------------------------------------------
# Load token-probe results
# ---------------------------------------------------------------------
comparison_results = {}

result_type = "VT training: ternary"

for step in STEPS_TO_COMPARE:

    result_path = WORKSPACE / (
        f"probe_results_ternary_step_{step}_{N_PROBE}_C_01_token_level.pkl"
    )

    if not result_path.exists():
        print(f"Missing: {result_path}")
        continue

    with open(result_path, "rb") as f:
        saved = pickle.load(f)

    token_result = (
        saved["token_probe"]
        if "token_probe" in saved
        else saved
    )

    comparison_results[(step, result_type)] = np.asarray(
        token_result["test_accuracy"],
        dtype=float,
    )

    print(
        f"Loaded step {step}:",
        comparison_results[(step, result_type)].shape,
    )

In [ ]:
print("\nAvailable results:")
for key in comparison_results:
    print(key)

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =====================================================================
# Checkpoints / stages
# =====================================================================
STEPS_TO_COMPARE = [
    512,
    5792,
    262144,
    1482912,
]

stage_labels = {
    512: "Stage 1",
    5792: "Stage 2",
    262144: "Stage 3",
    1482912: "Stage 4",
}

# Theoretical posterior accuracy associated with each checkpoint.
theoretical_accuracy = {
    512: 0.417308,
    5792: 0.507880,
    262144: 0.737,
    1482912: 0.851,
}


# =====================================================================
# Colors
# =====================================================================
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

colors = {
    step: color_cycle[i]
    for i, step in enumerate(STEPS_TO_COMPARE)
}


# =====================================================================
# Load empirical final-token probe results
# =====================================================================
token_probe_results = {}

for step in STEPS_TO_COMPARE:

    result_path = WORKSPACE / (
        f"probe_results_ternary_step_{step}_"
        f"{N_PROBE}_C_01_token_level.pkl"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            f"Missing final-token probe result:\n{result_path}"
        )

    with open(result_path, "rb") as f:
        saved = pickle.load(f)

    token_result = (
        saved["token_probe"]
        if "token_probe" in saved
        else saved
    )

    token_probe_results[step] = np.asarray(
        token_result["test_accuracy"],
        dtype=float,
    )


# =====================================================================
# Load third-last level-2 latent probe results
#
# This assumes the saved bundle has the structure
#
#     saved["results"][step]["probes"]["third_last_level2"]["test"]
#
# as in the earlier probing cell.
# =====================================================================
LEVEL2_RESULTS_PATH = (
    WORKSPACE / "second_third_last_probe_checkpoint_results.pkl"
)

if not LEVEL2_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing level-2 latent probe bundle:\n{LEVEL2_RESULTS_PATH}"
    )

with LEVEL2_RESULTS_PATH.open("rb") as f:
    latent_saved = pickle.load(f)

latent_results = latent_saved["results"]

level2_probe_results = {}

for step in STEPS_TO_COMPARE:

    if step not in latent_results:
        raise KeyError(
            f"Checkpoint t={step} not found in {LEVEL2_RESULTS_PATH.name}.\n"
            f"Available checkpoints: {list(latent_results.keys())}"
        )

    level2_probe_results[step] = np.asarray(
        latent_results[step]["probes"]["third_last_level2"]["test"],
        dtype=float,
    )


# =====================================================================
# Transformer depth
# =====================================================================
all_curves = (
    list(token_probe_results.values())
    + list(level2_probe_results.values())
)

n_depths = {len(acc) for acc in all_curves}

if len(n_depths) != 1:
    raise ValueError(
        f"Inconsistent numbers of transformer depths: {n_depths}"
    )

n_probe_points = next(iter(n_depths))
depth = np.arange(n_probe_points)

layer_labels = ["emb"] + [
    f"L{k}" for k in range(1, n_probe_points)
]

L3_INDEX = layer_labels.index("L3")


# =====================================================================
# Plot
# =====================================================================
fig, ax = plt.subplots(figsize=(8.0, 5.0))


# ---------------------------------------------------------------------
# Final-token probing curves
# ---------------------------------------------------------------------
for step in STEPS_TO_COMPARE:

    ax.plot(
        depth,
        token_probe_results[step],
        color=colors[step],
        linestyle="-",
        marker="o",
        linewidth=2.4,
        markersize=6,
        alpha=0.95,
        zorder=4,
    )


# ---------------------------------------------------------------------
# Third-last level-2 latent probing curves
# ---------------------------------------------------------------------
for step in STEPS_TO_COMPARE:

    ax.plot(
        depth,
        level2_probe_results[step],
        color=colors[step],
        linestyle="--",
        marker="o",
        linewidth=1.8,
        markersize=5,
        alpha=0.45,
        zorder=3,
    )


# ---------------------------------------------------------------------
# Theoretical posterior plateaus
# ---------------------------------------------------------------------
x_right = depth[-1]

for step in STEPS_TO_COMPARE:

    if step == 512:
        # Keep the original left-shifted blue plateau.
        x_start = x_right + 0.10 - 4
        x_end = x_right + 0.40 - 4
    else:
        x_start = x_right + 0.10
        x_end = x_right + 0.40

    ax.hlines(
        y=theoretical_accuracy[step],
        xmin=x_start,
        xmax=x_end,
        color=colors[step],
        linestyle="-",
        linewidth=2.6,
        zorder=5,
    )


# ---------------------------------------------------------------------
# Chance level
# ---------------------------------------------------------------------
ax.axhline(
    1.0 / V,
    color="grey",
    linestyle=":",
    linewidth=1.5,
    zorder=1,
)


# ---------------------------------------------------------------------
# Inside / Outside separation at L3
# ---------------------------------------------------------------------
ax.axvline(
    L3_INDEX,
    color="black",
    linestyle="--",
    linewidth=1.3,
    alpha=0.65,
    zorder=2,
)

# Labels placed high in the panel, on either side of L3.
ax.text(
    L3_INDEX - 0.18,
    0.965,
    "Inside",
    ha="right",
    va="top",
    fontsize=12,
)

ax.text(
    L3_INDEX + 0.18,
    0.965,
    "Outside",
    ha="left",
    va="top",
    fontsize=12,
)


# ---------------------------------------------------------------------
# Axes
# ---------------------------------------------------------------------
ax.set_xticks(
    depth,
    layer_labels,
)

ax.set_xlim(
    depth[0] - 0.2,
    depth[-1] + 0.55,
)

ax.set_ylim(0, 1.02)

ax.set_xlabel("Transformer depth")
ax.set_ylabel("Probe test accuracy")

ax.grid(alpha=0.2)


# =====================================================================
# Legends
# =====================================================================

# ---------------------------------------------------------------------
# Legend 1: what is being probed
# ---------------------------------------------------------------------
probe_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        marker="o",
        linewidth=2.4,
        markersize=6,
        label="Last token",
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        marker="o",
        linewidth=1.8,
        markersize=5,
        alpha=0.5,
        label="Level-2 latent",
    ),
]

legend_probe = ax.legend(
    handles=probe_handles,
    title="Probing accuracy of",
    loc="upper left",
    frameon=False,
)

ax.add_artist(legend_probe)


# ---------------------------------------------------------------------
# Legend 2: training stage / colour
# ---------------------------------------------------------------------
stage_handles = [
    Line2D(
        [0],
        [0],
        color=colors[step],
        linestyle="-",
        linewidth=2.6,
        label=stage_labels[step],
    )
    for step in STEPS_TO_COMPARE
]

ax.legend(
    handles=stage_handles,
    title="Training stage",
    loc="lower right",
    frameon=False,
)


# =====================================================================
# Final layout
# =====================================================================
plt.tight_layout()
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =====================================================================
# Plot style for a double-column paper figure
# =====================================================================
plt.rcParams.update({
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8,
    "legend.title_fontsize": 8.5,
    "lines.linewidth": 1.8,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


# =====================================================================
# Checkpoints / stages
# =====================================================================

# Final-token probe checkpoints
TOKEN_STEPS = [
    512,
    5792,
    262144,
    1482912,
]

# Corresponding level-2 latent probe checkpoints
LATENT_STEPS = {
    512: 512,
    5792: 5792,
    262144: 262144,
    1482912: 1_048_576,
}

stage_labels = {
    512: "Stage 1",
    5792: "Stage 2",
    262144: "Stage 3",
    1482912: "Stage 4",
}

# Theoretical posterior accuracy associated with each stage
theoretical_accuracy = {
    512: 0.417308,
    5792: 0.507880,
    262144: 0.737,
    1482912: 0.851,
}


# =====================================================================
# Colors
# =====================================================================
color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

colors = {
    step: color_cycle[i]
    for i, step in enumerate(TOKEN_STEPS)
}


# =====================================================================
# Load empirical final-token probe results
# =====================================================================
token_probe_results = {}

for step in TOKEN_STEPS:

    result_path = WORKSPACE / (
        f"probe_results_ternary_step_{step}_"
        f"{N_PROBE}_C_01_token_level.pkl"
    )

    if not result_path.exists():
        raise FileNotFoundError(
            f"Missing final-token probe result:\n{result_path}"
        )

    with open(result_path, "rb") as f:
        saved = pickle.load(f)

    token_result = (
        saved["token_probe"]
        if "token_probe" in saved
        else saved
    )

    token_probe_results[step] = np.asarray(
        token_result["test_accuracy"],
        dtype=float,
    )


# =====================================================================
# Load third-last level-2 latent probe results
# =====================================================================
LEVEL2_RESULTS_PATH = (
    WORKSPACE / "second_third_last_probe_checkpoint_results.pkl"
)

if not LEVEL2_RESULTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing level-2 latent probe bundle:\n{LEVEL2_RESULTS_PATH}"
    )

with LEVEL2_RESULTS_PATH.open("rb") as f:
    latent_saved = pickle.load(f)

latent_results = latent_saved["results"]

level2_probe_results = {}

for stage_step in TOKEN_STEPS:

    latent_step = LATENT_STEPS[stage_step]

    if latent_step not in latent_results:
        raise KeyError(
            f"Latent checkpoint t={latent_step} not found in "
            f"{LEVEL2_RESULTS_PATH.name}.\n"
            f"Available checkpoints: {list(latent_results.keys())}"
        )

    level2_probe_results[stage_step] = np.asarray(
        latent_results[latent_step]["probes"]["third_last_level2"]["test"],
        dtype=float,
    )


# =====================================================================
# Transformer depth
# =====================================================================
all_curves = (
    list(token_probe_results.values())
    + list(level2_probe_results.values())
)

n_depths = {len(acc) for acc in all_curves}

if len(n_depths) != 1:
    raise ValueError(
        f"Inconsistent numbers of transformer depths: {n_depths}"
    )

n_probe_points = next(iter(n_depths))
depth = np.arange(n_probe_points)

layer_labels = ["emb"] + [
    f"L{k}" for k in range(1, n_probe_points)
]

L3_INDEX = layer_labels.index("L3")


# =====================================================================
# Figure
#
# ~7 inches wide is a good target for a full-width/double-column figure.
# =====================================================================
fig, ax = plt.subplots(
    figsize=(7.0, 3.8)
)


# =====================================================================
# Final-token probing curves
# =====================================================================
for step in TOKEN_STEPS:

    ax.plot(
        depth,
        token_probe_results[step],
        color=colors[step],
        linestyle="-",
        marker="o",
        linewidth=1.9,
        markersize=4.5,
        alpha=0.95,
        zorder=4,
    )


# =====================================================================
# Third-last level-2 latent probing curves
# =====================================================================
for step in TOKEN_STEPS:

    ax.plot(
        depth,
        level2_probe_results[step],
        color=colors[step],
        linestyle="--",
        marker="o",
        linewidth=1.5,
        markersize=4,
        alpha=0.45,
        zorder=3,
    )


# =====================================================================
# Theoretical posterior plateaus
# =====================================================================
x_right = depth[-1]

for step in TOKEN_STEPS:

    if step == 512:
        # Original left-shifted Stage-1 plateau
        x_start = x_right + 0.10 - 4.5
        x_end = x_right + 0.40 - 4.5
    else:
        x_start = x_right + 0.10
        x_end = x_right + 0.40

    ax.hlines(
        y=theoretical_accuracy[step],
        xmin=x_start,
        xmax=x_end,
        color=colors[step],
        linestyle="-",
        linewidth=2.2,
        zorder=5,
    )


# =====================================================================
# Chance level
# =====================================================================
ax.axhline(
    1.0 / V,
    color="grey",
    linestyle=":",
    linewidth=1.2,
    zorder=1,
)


# =====================================================================
# Inside / Outside separation
# =====================================================================
ax.axvline(
    L3_INDEX,
    color="black",
    linestyle="--",
    linewidth=1.0,
    alpha=0.55,
    zorder=2,
)

ax.text(
    L3_INDEX - 0.18,
    0.97,
    "Inside",
    ha="right",
    va="top",
    fontsize=9,
)

ax.text(
    L3_INDEX + 0.18,
    0.97,
    "Outside",
    ha="left",
    va="top",
    fontsize=9,
)


# =====================================================================
# Axes
# =====================================================================
ax.set_xticks(
    depth,
    layer_labels,
)

ax.set_xlim(
    depth[0] - 0.2,
    depth[-1] + 0.55,
)

ax.set_ylim(0, 1.02)

ax.set_xlabel("Transformer depth")
ax.set_ylabel("Probe test accuracy")

ax.grid(
    alpha=0.18,
    linewidth=0.6,
)


# =====================================================================
# Legend 1: what is being probed
# =====================================================================
probe_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="-",
        marker="o",
        linewidth=1.9,
        markersize=4.5,
        label="Last token",
    ),
    Line2D(
        [0],
        [0],
        color="black",
        linestyle="--",
        marker="o",
        linewidth=1.5,
        markersize=4,
        alpha=0.5,
        label="Level-2 latent",
    ),
]

legend_probe = ax.legend(
    handles=probe_handles,
    title="Probing accuracy of",
    loc="upper left",
    frameon=False,
    handlelength=2.4,
)

ax.add_artist(legend_probe)


# =====================================================================
# Legend 2: training stages / colors
# =====================================================================
stage_handles = [
    Line2D(
        [0],
        [0],
        color=colors[step],
        linestyle="-",
        linewidth=2.0,
        label=stage_labels[step],
    )
    for step in TOKEN_STEPS
]

ax.legend(
    handles=stage_handles,
    title="Training stage",
    loc="lower right",
    frameon=False,
    handlelength=2.4,
)


# =====================================================================
# Layout and save
# =====================================================================
fig.tight_layout()

PDF_PATH = WORKSPACE / "token_and_level2_probe_across_depth.pdf"
PNG_PATH = WORKSPACE / "token_and_level2_probe_across_depth.png"

fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
)

fig.savefig(
    PNG_PATH,
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved PDF to {PDF_PATH}")
print(f"Saved PNG to {PNG_PATH}")

plt.show()